[![Open In Colab](https://Colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyneuro/Chapter_Colabs/blob/main/Colab_B.ipynb)

# Set B — Passive Membrane (RC & Cable)
**Author: Neural Engineering Laboratory, University of Missouri — Gregory Glickert, Khuram Choudhry, Ziao Chen, Satish S. Nair**

---

## Table of Contents
* [**B0:** Starter / Initialization](#B0)
* [**B1:** Nernst → Leak Reversal](#B1)
* [**B2:** Step Response & Time Constant $\tau$](#B2)
* [**B3:** Brief Pulse → Impulse Response](#B3)
* [**B4:** Sinusoidal Frequency Response](#B4)
* [**B5:** $R_{in}$ vs. $g_{pas}$ Relationship](#B5)
* [**B6:** Dendritic Attenuation & Space Constant $\lambda$](#B6)
* [**B7:** Estimating $\lambda$ from Steady-State Profile](#B7)
* [**B8:** Sealed-End Effects vs. Longer Cable](#B8)
* [**B9:** Temporal & Spatial Summation](#B9)
* [**B10:** Playground: Parameter Sweep](#B10)

---

<a id='B0'></a>
# B0 — Starter / Initialization

### Purpose
* **Construct a Passive Cable Model:** Initialize a spherical soma connected to a passive dendritic cable to study signal propagation.
* **Interactive Simulation:** Implement `ipywidgets` to allow real-time exploration of membrane time constants ($\tau$) and space constants ($\lambda$).
* **Dual-View Visualization:** Standardize a side-by-side plotting utility to compare temporal responses (Voltage vs. Time) and spatial attenuation (Voltage vs. Distance).

---

### Instructions for Students
1.  **Run the code cell below** to install the NEURON environment and load the interactive `run_interactive()` utility.
2.  **Verify the Environment:** Ensure the output confirms `NEURON 8.2.4` and that all libraries (Matplotlib, NumPy, ipywidgets) are loaded.
3.  **Note Global Defaults:** Baseline parameters for these simulations are:
    * **Leak Conductance ($g_{pas}$):** $0.0002\ S/cm^2$
    * **Specific Capacitance ($c_m$):** $1.0\ \mu F/cm^2$
    * **Axial Resistivity ($R_a$):** $100\ \Omega\cdot cm$

> **Reproducibility Note:** Always document your slider settings (e.g., $L$, $diam$, $g_{pas}$) when capturing data. A voltage trace is only meaningful if the underlying biophysical parameters are known.

---

### Post-Run Checklist
* [ ] **NEURON Version:** Confirm 8.2.4.
* [ ] **Sliders Visible:** Ensure the interactive control panel appears below the code output.

In [ ]:
#@title Run this cell to initialize the Lab { display-mode: "form" }
# --- Set C Initialization ---
# This cell is hidden to keep the workspace clean for students.

!pip -q install neuron==8.2.4
import ipywidgets as widgets
from neuron import h
from neuron.units import ms, mV
import matplotlib.pyplot as plt
import numpy as np

h.load_file("stdrun.hoc")

# Setup Morphology
soma = h.Section(name='soma')
soma.L = soma.diam = 20
soma.Ra = 100
soma.insert('pas')

dend = h.Section(name='dend')
dend.L = 800
dend.diam = 1.5
dend.Ra = 100
dend.insert('pas')
dend.connect(soma(1.0))

# Recorders
t_vec = h.Vector().record(h._ref_t)
v_soma = h.Vector().record(soma(0.5)._ref_v)
v_dend = h.Vector().record(dend(1.0)._ref_v)

# MINIMAL CHANGE: Added default parameter e_pas=-65 to allow B1 updates without breaking defaults
def run_interactive(g_pas=0.0002, cm=1.0, stim_amp=-0.05, dend_L=800, e_pas=-65):
    for sec in h.allsec():
        sec.g_pas = g_pas
        sec.cm = cm
        sec.e_pas = e_pas   # MINIMAL CHANGE: Use parameter instead of hardcoded -65

    dend.L = dend_L

    stim = h.IClamp(dend(1.0))
    stim.delay, stim.dur, stim.amp = 5, 40, stim_amp

    h.finitialize(e_pas * mV) # MINIMAL CHANGE: Finitialize at dynamic resting potential
    h.continuerun(70 * ms)

    # RETAIN ORIGINAL DUAL-PANEL LAYOUT
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(t_vec, v_soma, 'k', label='Soma')
    ax1.plot(t_vec, v_dend, 'r--', label='Distal Tip')
    ax1.set_title('Temporal Response'); ax1.set_xlabel('Time (ms)'); ax1.set_ylabel('Vm (mV)')
    ax1.legend(); ax1.grid(True, alpha=0.3)

    xs = np.linspace(0, 1, 11)
    vs = [dend(x).v for x in xs]
    ax2.plot(xs * dend.L, vs, 'bo-')
    ax2.set_title('Spatial Attenuation'); ax2.set_xlabel('Distance (µm)'); ax2.set_ylabel('Steady V (mV)')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout(); plt.show()

print('Interactive Set C Environment Ready.')

<a id='C1'></a>
## B1 — Nernst → Leak Reversal (Passive $E_L$)

### The Concept
Physiological concentrations define ion-specific **Nernst Potentials**. In a passive model, the "Leak Reversal" ($e_{pas}$) represents the weighted average of all open channels at rest. By setting $e_{pas}$ to a specific ion's Nernst potential, we can simulate a membrane dominated by that single ion species.

### Exercises
1.  **Temperature Dependence:** Compute the Nernst potential for $K^+$ at $27^\circ C$ and $37^\circ C$. Explain why the value increases with temperature.
2.  **Driving Force:** If you set $e_{pas} = -90\ mV$ but the cell is currently at $-65\ mV$, which direction will the positive current flow?

In [ ]:
# @title B1: Interactive Nernst Calculator { display-mode: "form" }
# @markdown Adjust the concentrations and temperature to calculate the equilibrium potential.
# @markdown The plot will update to show the new resting membrane potential.

import numpy as np
import ipywidgets as widgets

def nernst_calculation(cin, cout, temp_c):
    # Constants
    R, F, z = 8.314, 96485, 1
    temp_k = 273.15 + temp_c

    # Nernst Equation
    val = (1000.0 * R * temp_k / (z * F)) * np.log(cout / cin)

    print(f"Calculated E_K: {val:.2f} mV at {temp_c}°C")

    # Update the interactive model to show the new 'Resting' state
    try:
        # MINIMAL CHANGE: Pass calculated val to e_pas
        run_interactive(g_pas=0.0002, cm=1.0, stim_amp=0.0, e_pas=val)
    except NameError:
        print("Error: Please run the B0 Initialization cell first!")

# Interactive Sliders
widgets.interact(nernst_calculation,
    cin=widgets.IntSlider(value=140, min=1, max=200, description='[K] in (mM)'),
    cout=widgets.IntSlider(value=5, min=1, max=50, description='[K] out (mM)'),
    temp_c=widgets.FloatSlider(value=37.0, min=0.0, max=40.0, step=1.0, description='Temp (°C)')
)

<a id='B2'></a>
## B2 — Step Response & Time Constant $\tau$

### The Concept
A passive membrane behaves like an **RC circuit**. When you inject a step of current, the voltage rises exponentially. The **Time Constant ($\tau$)** is the time it takes for the membrane to reach **~63.2%** of its final steady-state value.

### What to Change
* **Capacitance ($c_m$):** Increase this to see the "slowdown" of the membrane.
* **Leak Conductance ($g_{pas}$):** Increase this to see how it affects both the speed and the final height of the voltage.

### Exercises
1.  **Measuring $\tau$:** Set $c_m = 1.0$ and $g_{pas} = 0.0002$. Find the time point where the voltage reaches 63% of its peak.
2.  **Calculation:** Given that $\tau = c_m / g_{pas}$ (in units of $ms$ when scaled), calculate the theoretical $\tau$ for the default values. Does it match the graph?

In [ ]:
# @title B2 Interactive Simulation { display-mode: "form" }
# @markdown Run this cell to show the sliders. You can hide the code by selecting 'Form > Hide code'.

import ipywidgets as widgets

def run_b2_step(g_pas, cm):
    try:
        run_interactive(
            g_pas=g_pas,
            cm=cm,
            stim_amp=-0.05,
            dend_L=800
        )
        # MINIMAL CHANGE: Fixed inverted scaling formula for print statement accuracy
        tau_val = (cm / g_pas) * 1000
        print(f"Theoretical Tau: {tau_val:.2f} ms")

    except NameError:
        print("Error: Please run the B0 Initialization cell first!")

widgets.interact(run_b2_step,
    g_pas=widgets.FloatLogSlider(value=0.0002, base=10, min=-5, max=-3, step=0.1, description='g_pas'),
    cm=widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='Cm')
)

<a id='B3'></a>
## B3 — Brief Pulse → Impulse-like Kernel

### The Concept
A very brief current pulse approximates the **impulse response** (temporal kernel) of the passive membrane. This represents how the membrane "filters" fast, discrete inputs. Because the membrane has capacitance, the voltage does not return to baseline instantly; it decays according to the time constant $\tau$.

### What to Change
* **Pulse Dynamics:** Adjust the pulse width and amplitude.
* **Initial State:** Observe the decay back to the baseline level ($-65\ mV$).

### Checks
1.  **Linearity Check:** Confirm that for small pulses, the peak voltage scales proportionally with the pulse amplitude. This verifies you are operating in the membrane's linear integration region.

### Exercises
1.  **Charge Conservation:** Halve the pulse width while keeping the total injected charge constant (i.e., double the amplitude). Report any changes in the peak voltage and the width of the resulting response.
2.  **Synaptic Integration:** Explain how this impulse-like kernel relates to EPSP shaping. Specifically, how would the membrane response differ for synapses with fast vs. slow time courses?

In [ ]:
# @title B3: Interactive Impulse Response { display-mode: "form" }
# @markdown Run this cell to toggle the interactive impulse response. You can hide the code
# @markdown by right-clicking and selecting 'Form > Hide code'.

import ipywidgets as widgets

def run_B3_pulse(width, amp):
    try:
        # We use the distal tip for consistency with the cable exercises
        # or soma(0.5) for a pure RC response. Let's use soma for the kernel.
        stim = h.IClamp(soma(0.5))
        stim.delay, stim.dur, stim.amp = 5, width, amp

        h.finitialize(-65 * mV)
        h.continuerun(50 * ms)

        plt.figure(figsize=(7, 3.5))
        plt.plot(t_vec, v_soma, 'k', label=f'Width: {width}ms')
        plt.axvspan(5, 5+width, color='gray', alpha=0.2, label='Stimulus Window')
        plt.title('B3: Impulse-like Response (Temporal Kernel)')
        plt.xlabel('Time (ms)'); plt.ylabel('Vm (mV)')
        plt.legend(); plt.grid(True, alpha=0.3); plt.show()
    except NameError:
        print("Error: Please run the B0 Initialization cell first!")

widgets.interact(run_B3_pulse,
    width=widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='Width (ms)'),
    amp=widgets.FloatSlider(value=-0.2, min=-1.0, max=0.0, step=0.05, description='Amp (nA)')
)

<a id='C4'></a>
## B4 — Sinusoidal Frequency Response

### The Concept
The passive membrane acts as a **first-order low-pass filter**. High-frequency inputs (e.g., 100 Hz) are attenuated more than low-frequency ones (e.g., 2 Hz). This occurs because the membrane capacitance ($C_m$) requires time to charge; at high frequencies, the current reverses direction before the membrane can reach its full potential.

### What to Change
* **Frequency:** Use the dropdown to cycle through frequencies.
* **Observe Attenuation:** Watch how the peak-to-peak voltage shrinks as frequency increases.
* **Observe Phase Lag:** Notice the delay between the peak of the input and the peak of the response.

### Exercises
1.  **Steady-State Metrics:** Record the peak-to-peak amplitude at 2, 10, and 50 Hz. Does the attenuation increase monotonically?
2.  **The -3dB Point:** In a low-pass filter, the "cutoff frequency" is where the power drops by half (amplitude drops to ~70.7% of the maximum). Based on your observations, estimate this frequency for the default model.
3.  **Low-Pass Logic:** Why is a 100 Hz signal attenuated more than a 10 Hz signal in a purely passive cell?

In [ ]:
# @title B4: Interactive Sine Frequency Sweep { display-mode: "form" }
# @markdown Run this cell to select different frequencies. The code is hidden to focus
# @markdown on the low-pass filtering effect.

import ipywidgets as widgets
import numpy as np

def run_b4_sine(freq_hz):
    try:
        dur_ms = 400
        # Setup the Sine Wave Input
        ic = h.IClamp(soma(0.5))
        ic.delay, ic.dur = 0, 1e9

        tt = np.arange(0, dur_ms, 0.1)
        # 0.05 nA amplitude sine wave
        ii = 0.05 * np.sin(2 * np.pi * freq_hz * (tt/1000.0))
        ivec, t_input = h.Vector(ii), h.Vector(tt)
        ivec.play(ic._ref_amp, t_input, 1)

        h.finitialize(-65 * mV)
        h.continuerun(dur_ms * ms)

        plt.figure(figsize=(10, 4))
        plt.plot(t_vec, v_soma, 'k', label='Membrane Potential')
        plt.title(f'B4: Sinusoidal Response at {freq_hz} Hz')
        plt.xlabel('Time (ms)'); plt.ylabel('Vm (mV)')

        # Zoom into the last few cycles to see phase lag clearly
        cycle_period = 1000.0 / freq_hz
        plt.xlim(dur_ms - (cycle_period * 3), dur_ms)

        plt.grid(True, alpha=0.3)
        plt.legend(loc='upper right')
        plt.show()

    except NameError:
        print("Error: Please run the B0 Initialization cell first!")

# Dropdown for specific frequencies to match the original lesson plan
widgets.interact(run_b4_sine, freq_hz=[2, 5, 10, 20, 50, 100, 200])

<a id='C5'></a>
## B5 — $R_{in}$ vs. $g_{pas}$ Relationship

### The Concept
**Input Resistance ($R_{in}$)** is a measure of how much the membrane potential changes in response to a given amount of current ($R = \Delta V / \Delta I$). In a passive cell, $R_{in}$ is inversely proportional to the **Leak Conductance ($g_{pas}$)**.

### What to Change
* **Leakiness:** Adjust $g_{pas}$ to see how "stiff" or "responsive" the membrane becomes.
* **Stimulus:** Change the current amplitude to verify that the relationship remains linear.

### Exercises
1.  **Inverse Relation:** Using the sliders, find the steady-state $\Delta V$ for $g_{pas} = 0.0001$ and $g_{pas} = 0.0005$. Calculate $R_{in}$ for both.
2.  **Numerical Validation:** For two different $g_{pas}$ values, check if the relationship $\tau = R_{in} \cdot C_m$ holds numerically. (Note: You may need to account for the surface area of the soma).
3.  **EPSP Scaling:** Why does a neuron with a **higher** $R_{in}$ exhibit larger voltage changes when subjected to the same synaptic current compared to a "leaky" neuron?

In [ ]:
# @title B5: Input Resistance vs. Conductance { display-mode: "form" }
# @markdown Adjust g_pas to see how it dictates the steady-state voltage change (Ohm's Law).

import ipywidgets as widgets

def run_b5_rin(g_pas, stim_amp):
    try:
        # Focus on the soma for a clear calculation of Rin
        for sec in h.allsec():
            sec.g_pas = g_pas
            sec.e_pas = -65

        stim = h.IClamp(soma(0.5))
        stim.delay, stim.dur, stim.amp = 5, 60, stim_amp

        h.finitialize(-65 * mV)
        h.continuerun(80 * ms)

        # Calculate Delta V at steady state (near the end of pulse)
        v_steady = v_soma.as_numpy()[-1]
        dv = v_steady - (-65)
        rin = abs(dv / stim_amp) if stim_amp != 0 else 0

        plt.figure(figsize=(7, 4))
        plt.plot(t_vec, v_soma, 'k')
        plt.axhline(v_steady, color='r', linestyle='--', alpha=0.5)
        plt.title(f'B5: Rin Exploration (Measured Rin ≈ {rin:.1f} MΩ)')
        plt.xlabel('Time (ms)'); plt.ylabel('Vm (mV)')
        plt.grid(True, alpha=0.3); plt.show()

    except NameError:
        print("Error: Please run the B0 Initialization cell first!")

widgets.interact(run_b5_rin,
    g_pas=widgets.FloatLogSlider(value=0.0002, base=10, min=-5, max=-3, step=0.1, description='g_pas'),
    stim_amp=widgets.FloatSlider(value=-0.05, min=-0.2, max=0.2, step=0.01, description='Stim Amp')
)

<a id='B6'></a>
## B6 — Dendritic Attenuation & Space Constant $\lambda$

### The Concept
A passive dendrite behaves as a **leaky cable**. As a signal propagates away from the site of current injection, the membrane potential ($V_m$) attenuates (decays). This spatial decay is characterized by the **Space Constant ($\lambda$)**, which is the distance at which the voltage drops to ~37% of its maximum value.

$$\lambda = \sqrt{\frac{r_m}{r_a}} = \sqrt{\frac{d \cdot R_m}{4 \cdot R_a}}$$

### What to Change
* **Diameter ($diam$):** See how making the "pipe" wider allows the signal to travel further.
* **Axial Resistivity ($R_a$):** Observe how increasing internal resistance "chokes" the signal propagation.
* **Leakiness ($g_{pas}$):** Watch the signal vanish faster as you make the membrane leakier.

### Exercises
1.  **Measuring $\lambda$:** With default parameters ($diam=1.5$, $Ra=100$, $g_{pas}=0.0002$), use the **Spatial Attenuation** plot (right side) to find the distance where the voltage is 37% of the peak at $x=0$.
2.  **Diameter Influence:** Double the diameter. Does the signal travel further? Mathematically, by what factor should $\lambda$ increase when diameter doubles?
3.  **Experimental Setup:** If you inject current into the soma instead of the distal tip, how does the attenuation profile change?

In [ ]:
# @title B6: Interactive Cable Properties { display-mode: "form" }
# @markdown Adjust morphology and cable properties to see the effect on spatial attenuation.

import ipywidgets as widgets

def run_b6_cable(diam, Ra, g_pas):
    try:
        # Update morphology and biophysics
        dend.diam = diam
        for sec in h.allsec():
            sec.Ra = Ra
            sec.g_pas = g_pas
            sec.e_pas = -65

        # Run the standard side-by-side simulation
        # Injecting at distal tip (dend(1.0)) to see propagation back to soma
        run_interactive(g_pas=g_pas, cm=1.0, stim_amp=-0.05, dend_L=800)

        # Theoretical Lambda calculation for the student
        rm = 1.0 / g_pas
        theoretical_lambda = np.sqrt((diam * 1e-4 * rm) / (4 * Ra)) * 10000 # in um
        print(f"Theoretical Space Constant (λ): {theoretical_lambda:.1f} µm")

    except NameError:
        print("Error: Please run the B0 Initialization cell first!")

widgets.interact(run_b6_cable,
    diam=widgets.FloatSlider(value=1.5, min=0.1, max=5.0, step=0.1, description='Diam (µm)'),
    Ra=widgets.FloatSlider(value=100, min=20, max=500, step=10, description='Ra (Ω·cm)'),
    g_pas=widgets.FloatLogSlider(value=0.0002, base=10, min=-5, max=-3, step=0.1, description='g_pas')
)

<a id='B7'></a>
## B7 — Estimate $\lambda$ from Steady-State Profile

### The Concept
At steady state, the voltage decay along a passive cable is approximately exponential:
$$V(x) = V_{rest} + (V_0 - V_{rest}) \cdot e^{-x/\lambda}$$

If we plot the natural log of the voltage change ($\ln |V(x) - V_{rest}|$) against the distance ($x$), the relationship becomes a straight line. The **slope** of this line is equal to $-1/\lambda$.

### What to Change
* **Morphology:** Change the diameter or length of the dendrite.
* **Biophysics:** Adjust $R_a$ or $g_{pas}$ and observe how the "straightness" and "slope" of the log-plot change.

### Exercises
1.  **Linear Regression:** Using the plot below, identify the slope of the line. Calculate $\lambda$ as $|1/slope|$.
2.  **Geometric Influence:** Halve the dendrite diameter. How does the slope of the log-plot change? Does this indicate a larger or smaller $\lambda$?
3.  **Boundary Effects:** Note if the line remains perfectly straight near the ends of the cable. Why might the "Sealed End" (the tip of the dendrite) cause a deviation from a perfect exponential decay?

In [ ]:
# @title B7: Linearizing Exponential Decay { display-mode: "form" }
# @markdown Run this cell to visualize the steady-state profile on a log scale.
# @markdown This allows for the direct estimation of the space constant.

import ipywidgets as widgets

def run_b7_log_profile(diam, g_pas, Ra):
    try:
        # Apply parameters
        dend.diam = diam
        for sec in h.allsec():
            sec.Ra = Ra
            sec.g_pas = g_pas
            sec.e_pas = -65

        # Stimulate at soma to see decay down the dendrite
        stim = h.IClamp(soma(0.5))
        stim.delay, stim.dur, stim.amp = 5, 100, 0.1

        h.finitialize(-65 * mV)
        h.continuerun(110 * ms)

        # Collect spatial data
        xs = np.linspace(0, 1, 21)
        dists = xs * dend.L
        vs = np.array([dend(x).v for x in xs])
        v_change = np.abs(vs - (-65))

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

        # Linear Scale
        ax1.plot(dists, vs, 'bo-')
        ax1.set_title('Steady-State Voltage')
        ax1.set_xlabel('Distance (µm)'); ax1.set_ylabel('Vm (mV)')
        ax1.grid(True, alpha=0.3)

        # Semi-Log Scale (The "Researcher's View")
        # We avoid log(0) by ensuring v_change > 0
        ax2.plot(dists, np.log(v_change + 1e-9), 'rs-')
        ax2.set_title('Semi-Log Plot: ln|ΔV| vs. Distance')
        ax2.set_xlabel('Distance (µm)'); ax2.set_ylabel('ln|ΔV|')
        ax2.grid(True, alpha=0.3)

        plt.tight_layout(); plt.show()

    except NameError:
        print("Error: Please run the B0 Initialization cell first!")

widgets.interact(run_b7_log_profile,
    diam=widgets.FloatSlider(value=1.5, min=0.5, max=5.0, step=0.5, description='Diam (µm)'),
    g_pas=widgets.FloatLogSlider(value=0.0002, base=10, min=-5, max=-3, description='g_pas'),
    Ra=widgets.FloatSlider(value=100, min=50, max=500, step=50, description='Ra (Ω·cm)')
)

<a id='B8'></a>
## B8 — Sealed-End Effects vs. Longer Cable

### The Concept
Boundary conditions matter. When a signal propagates to the end of a dendrite, it encounters a "Sealed End" (a high-resistance boundary). Because the current cannot continue further down the cable, it "piles up," causing the voltage at the terminal to be higher than it would be in a longer, continuous cable.

### What to Change
* **Dendrite Length ($L$):** Compare a "Short" dendrite ($L < \lambda$) to a "Long" dendrite ($L > 3\lambda$).
* **Observation Point:** Notice the difference in voltage at the distal tip versus the soma.

### Exercises
1.  **Terminal Boost:** Set the dendrite length to $200\ \mu m$. Inject current at the soma and record the voltage at the tip. Now increase the length to $1000\ \mu m$. Why does the voltage at the $200\ \mu m$ mark drop when the cable is made longer?
2.  **The Infinite Cable Approximation:** At what length (relative to $\lambda$) does the cable start behaving like it has no end?
3.  **Functional Significance:** Why might a sensory neuron evolve a "sealed end" at its distal dendrite rather than letting the dendrite continue?

In [ ]:
# @title B8: Sealed-End vs. Infinite Cable { display-mode: "form" }
# @markdown Experiment with the total length of the cable to see how boundary conditions
# @markdown affect the voltage profile.

import ipywidgets as widgets

def run_b8_boundaries(dend_L, stim_loc):
    try:
        # Reset parameters to standard
        for sec in h.allsec():
            sec.Ra = 100
            sec.g_pas = 0.0002
            sec.e_pas = -65

        dend.L = dend_L

        # Stimulate based on user choice
        loc = 0.0 if stim_loc == 'Soma' else 1.0
        stim = h.IClamp(soma(0.5) if loc == 0.0 else dend(1.0))
        stim.delay, stim.dur, stim.amp = 5, 100, 0.1

        h.finitialize(-65 * mV)
        h.continuerun(110 * ms)

        # Spatial Profile
        xs = np.linspace(0, 1, 50)
        vs = [dend(x).v for x in xs]

        plt.figure(figsize=(8, 4))
        plt.plot(xs * dend_L, vs, 'g-', lw=2, label=f'Length = {dend_L}µm')
        plt.axhline(-65, color='k', linestyle=':', alpha=0.3)
        plt.title(f'B8: Spatial Profile (Stimulating at {stim_loc})')
        plt.xlabel('Distance from Soma (µm)'); plt.ylabel('Vm (mV)')
        plt.grid(True, alpha=0.3); plt.legend(); plt.show()

    except NameError:
        print("Error: Please run the B0 Initialization cell first!")

widgets.interact(run_b8_boundaries,
    dend_L=widgets.IntSlider(value=400, min=50, max=2000, step=50, description='Cable L (µm)'),
    stim_loc=['Soma', 'Distal Tip']
)

<a id='B9'></a>
## B9 — Temporal & Spatial Summation

### The Concept
Neurons rarely receive just one input.
1.  **Temporal Summation:** Occurs when a single synapse fires in rapid succession. If the interval is shorter than the time constant ($\tau$), the second response builds upon the "tail" of the first.
2.  **Spatial Summation:** Occurs when multiple synapses at different locations fire simultaneously. The inputs converge at the soma, but their individual contributions are filtered by their respective space constants ($\lambda$).

### What to Change
* **Interval:** Change the time between pulses to see the "summation window."
* **Location:** Move the second input further away to see how distance weakens spatial cooperation.

### Exercises
1.  **The Summation Window:** Using the Temporal slider, find the maximum interval (in ms) where you can still see at least a 10% "boost" on the second peak compared to the first. How does this relate to $\tau$?
2.  **Sublinear Summation:** Fire two distal inputs simultaneously. Is the peak at the soma exactly $V_1 + V_2$, or is it slightly less? (Hint: Look up "driving force" and "shunting").
3.  **Non-Linearity:** Why might two inputs on the same dendritic branch sum differently than two inputs on different branches?

In [ ]:
# @title B9: Interactive Summation { display-mode: "form" }
# @markdown Toggle between Temporal and Spatial summation modes to see how inputs combine.

import ipywidgets as widgets

def run_b9_summation(mode, interval_or_dist, amp):
    try:
        # Clear previous settings
        h.finitialize(-65 * mV)

        if mode == 'Temporal (Soma)':
            # Train of 3 pulses at the soma
            stims = []
            for i in range(3):
                s = h.IClamp(soma(0.5))
                s.delay = 10 + (i * interval_or_dist)
                s.dur = 2
                s.amp = amp
                stims.append(s)
            title_str = f"Temporal Summation (Interval: {interval_or_dist}ms)"

        else: # Spatial Summation
            # One input at soma, one at variable distance on dendrite
            s1 = h.IClamp(soma(0.5))
            s1.delay, s1.dur, s1.amp = 10, 5, amp

            # interval_or_dist here acts as the 'x' location (0 to 1)
            loc = interval_or_dist / 1000.0 # scale slider to 0-1
            s2 = h.IClamp(dend(min(loc, 1.0)))
            s2.delay, s2.dur, s2.amp = 10, 5, amp
            stims = [s1, s2]
            title_str = f"Spatial Summation (Soma + Dendrite @ {interval_or_dist}µm)"

        h.continuerun(100 * ms)

        plt.figure(figsize=(8, 4))
        plt.plot(t_vec, v_soma, 'k', lw=2)
        plt.title(title_str)
        plt.xlabel('Time (ms)'); plt.ylabel('Vm (mV)')
        plt.grid(True, alpha=0.3); plt.show()

    except NameError:
        print("Error: Please run the B0 Initialization cell first!")

widgets.interact(run_b9_summation,
    mode=['Temporal (Soma)', 'Spatial (Convergent)'],
    interval_or_dist=widgets.IntSlider(value=10, min=1, max=100, description='Interval/Dist'),
    amp=widgets.FloatSlider(value=0.1, min=0.01, max=0.5, step=0.01, description='Amp (nA)')
)

<a id='B10'></a>
## B10 — Playground: Parameter Sweep for $\tau$, $R_{in}$, and Attenuation

### The Objective
Systematically observe how biophysical properties ($c_m, g_{pas}$) and morphology ($diam$) interact to define the "electrical reach" and "speed" of a neuron.

### Conceptual Checks
* **Capacitance:** $\uparrow c_m \rightarrow \uparrow \tau$ (Slower membrane charging).
* **Leak:** $\uparrow g_{pas} \rightarrow \downarrow R_{in}$ (Lower input resistance/gain).
* **Diameter:** $\uparrow diameter \rightarrow \downarrow attenuation$ (Better signal propagation).

### Exercises
1. **The "Leaky" vs. "Tight" Membrane:** Compare the **Attenuation Index** (distal/somatic voltage ratio) for $g_{pas} = 0.0001$ vs $0.0005$. Which one allows the soma to "hear" the dendrite better?
2. **Coincidence Detection Constraints:** In two sentences, explain how the time constant ($\tau$) and the space constant ($\lambda$) jointly constrain the "window" for coincidence detection.
3. **Attenuation Index:** Use the automated calculation in the simulation below to report the index for a "thick" ($3\ \mu m$) vs. "thin" ($0.5\ \mu m$) dendrite.

In [ ]:
# @title B10: Parameter Sweep Playground { display-mode: "form" }
# @markdown Use this "Global Controller" to explore the trade-offs between speed,
# @markdown gain, and spatial reach.

import ipywidgets as widgets
import numpy as np

def run_B10_playground(cm, g_pas, diam):
    try:
        # We set Ra to 100 internally to avoid the 'widgets.Fixed' error
        Ra_internal = 100

        # Update all compartments with the new slider values
        for sec in h.allsec():
            sec.cm = cm
            sec.g_pas = g_pas
            sec.Ra = Ra_internal
            sec.e_pas = -65
        dend.diam = diam

        # Run the standard side-by-side simulation
        # Injecting at distal tip to measure attenuation back to soma
        run_interactive(g_pas=g_pas, cm=cm, stim_amp=-0.05, dend_L=800)

        # Calculate the Attenuation Index (V_soma_max / V_dend_max)
        v_soma_max = np.max(np.abs(v_soma.as_numpy() - (-65)))
        v_dend_max = np.max(np.abs(v_dend.as_numpy() - (-65)))
        atten_index = v_soma_max / v_dend_max if v_dend_max > 0 else 0

        # Calculate theoretical Tau for the summary
        tau_ms = (cm / (g_pas * 1000)) * 10

        print(f"\n--- Laboratory Summary Stats ---")
        print(f"Membrane Time Constant (τ): {tau_ms:.2f} ms")
        print(f"Attenuation Index (Soma/Distal): {atten_index:.3f}")
        print(f"(Closer to 1.0 = More efficient propagation)")

    except NameError:
        print("Error: Please run the B0 Initialization cell first!")

# Removed widgets.Fixed to prevent the AttributeError
widgets.interact(run_b10_playground,
    cm=widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='Cm (uF)'),
    g_pas=widgets.FloatLogSlider(value=0.0002, base=10, min=-5, max=-3, description='g_pas'),
    diam=widgets.FloatSlider(value=1.5, min=0.2, max=5.0, step=0.1, description='Diam (um)')
)

## 🧠 Set B — Quick-Fire Review
*Before diving into the long-form questions, check your intuition against these findings from the simulations:*

* **The "Speed" Rule:** If you want a neuron to respond faster to a synapse, should you **decrease** its membrane capacitance ($c_m$)?
* **The "Reach" Rule:** To make a distal EPSP travel further toward the soma without dying out, should the dendrite be **thick** or **thin**?
* **The "Filter" Rule:** Which frequency is harder for a passive membrane to "track" without losing amplitude: **5 Hz** or **50 Hz**?
* **The "Boundary" Rule:** Why does a signal seem to "bounce" or get larger at the very end of a sealed dendrite?

---

## 📝 Practice & Discussion Questions

1. Define $\tau$ and $\lambda$ in words, and give one empirical method to estimate each.
2. **Predict:** If dendritic diameter decreases, what happens to $\lambda$ and distal EPSP amplitude at the soma?
3. Why does an EPSP recorded locally look different from the somatic EPSP for the same input event?
4. Show qualitatively how branch points can affect attenuation.
5. Explain why the passive membrane behaves like a **low-pass filter**; connect this to synaptic frequency content.
6. **Reasoning:** How does increasing membrane resistance affect $\tau$ and EPSP area?
7. Give one case where apparent linear summation fails and what it suggests mechanistically.
8. Provide one realistic parameter set ($R_m, C_m, R_a$) and describe the expected EPSP shape qualitatively.
9. Why might a **distal** EPSP be broader at the soma than a **proximal** EPSP?
10. **Compute:** If $\tau$ doubles, what happens to EPSP half-width at the soma (qualitatively)?
11. Explain **electrotonic distance** vs. physical distance.
12. Identify a diagnostic plot (e.g., semi-log or transfer impedance) that reveals cable effects.
13. Why can passive properties alone **mislead** scientists about synaptic strength across different locations?
14. **Design:** A single-compartment model fits soma data but not dendritic data. What is the next minimal step?
15. What are the limits of passive models, and what does adding active conductances (voltage-gated channels) provide?
16. Explain how $\tau$ and $\lambda$ together capture the difference between **temporal** vs. **spatial** integration.
17. Describe how **holding potential** can change apparent PSP size in different compartments.
18. Provide one artifact that could confound cable measurements and how to control it.
19. **Synthesis:** State the minimal passive features to keep for Set E timing experiments.
20. **Interactive Validation:** In B10, you observed the "Attenuation Index." Explain why this index is a more useful measure of "synaptic weight" than just measuring the voltage at the synapse itself.
21. **The Nernst-Leak Link:** If a neuron's temperature increases from 20°C to 37°C, how does that shift the resting membrane potential? Refer to your findings in B1.
22. **Clinical Connection:** Multiple Sclerosis involves the loss of myelin (which acts like decreasing $R_m$). Based on your B6 results, how does losing $R_m$ affect the space constant $\lambda$ and the reliability of signal propagation?
23. **Final Summary:** Summarize how Set C prepares you for studying timing motifs and location dependence in more complex neural circuits.

---

## 📋 Quick Reference: Passive Parameter Cheat Sheet

| Parameter | Symbol | Effect of **Increasing** Value | Result on Signal |
| :--- | :--- | :--- | :--- |
| **Capacitance** | $c_m$ | Increases $\tau$ | **Slower** temporal response |
| **Leak Conductance** | $g_{pas}$ | Decreases $R_{in}$ and $\lambda$ | **Smaller & shorter** reach |
| **Diameter** | $diam$ | Increases $\lambda$ | **Further** spatial reach |
| **Axial Resistance**| $R_a$ | Decreases $\lambda$ | **Higher** internal signal loss |